# Lección 6.5 — Stacking y voting: un comité de modelos distintos

> **Módulo 6 · Ensambles: la sabiduría de la multitud** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 60 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 6.1 (por qué combinar modelos), Lecciones 5.2–5.6 (modelos clásicos), Lección 4.6 (validación cruzada), Lección 2.6 (fuga de datos)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Distinguir el **voto duro** (*hard voting*) del **voto suave** (*soft voting*) y calcular ambos a mano.
- Explicar qué es **stacking**: un meta-modelo que aprende **a quién creerle**.
- Explicar por qué el meta-modelo debe aprender con predicciones **fuera de pliegue** (*out-of-fold*) y qué pasa si no.
- Usar `VotingClassifier` y `StackingClassifier` y decidir cuándo valen la pena.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor, HistGradientBoostingRegressor,
                              VotingClassifier, StackingClassifier, VotingRegressor, StackingRegressor)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.datasets import make_classification, load_diabetes

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el jurado de un concurso de cocina

Un concurso de cocina tiene tres jueces: una **pastelera**, un **parrillero** y un **crítico** de restaurantes. Cada uno sabe de cosas distintas: es el comité **diverso** de la Lección 6.1. ¿Cómo combinan sus opiniones?

| Forma | Cómo decide | En ML |
|---|---|---|
| **Voto a mano alzada** | Cada juez levanta "✅ pasa" o "❌ no pasa". Gana la mayoría | **Voto duro** (*hard voting*) |
| **Puntaje** | Cada juez da una nota de 0 a 10 y se promedian | **Voto suave** (*soft voting*): se promedian **probabilidades** |
| **El jefe de cocina** | Un chef que lleva años viendo a estos jueces aprendió *"en postres le creo a la pastelera, en carnes al parrillero"* | **Stacking**: un **meta-modelo** aprende a combinar a los demás |

A diferencia de bagging y boosting (Lecciones 6.2–6.4), aquí los miembros suelen ser modelos **de familias distintas**: una regresión logística, un KNN, un bosque…

### ✋ Ejercicio 1 — Voto duro contra voto suave
Tres modelos estiman la probabilidad de que un paciente esté enfermo:

| Modelo | Probabilidad de "enfermo" |
|---|---|
| A | 0.95 |
| B | 0.40 |
| C | 0.45 |

1. Con **voto duro** (cada modelo dice "enfermo" si su probabilidad es ≥ 0.5), ¿qué decide el comité?
2. Con **voto suave** (promedio de probabilidades, "enfermo" si el promedio ≥ 0.5), ¿qué decide?
3. ¿Cuál te parece más razonable aquí y por qué?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. A dice "enfermo"; B y C dicen "sano" → **sano** (2 contra 1).
2. Promedio = (0.95 + 0.40 + 0.45) / 3 = **0.60** → **enfermo**.
3. El voto suave, en general: A está **muy seguro**, mientras que B y C apenas dudan hacia "sano" (0.40 y 0.45 están cerca de 0.5). El voto duro trata igual un "no" tímido que un "sí" rotundo; el voto suave **escucha también la seguridad** de cada modelo. Condición: que las probabilidades de los modelos sean razonables (bien calibradas).
</details>

---
## 2. 📖 Voting en la práctica

Usaremos 1,500 ejemplos sintéticos con 8 características, y cuatro miembros de familias distintas:
- **Regresión logística** (Lección 5.2): traza una frontera recta.
- **KNN** con k=15 (Lección 5.3): mira a los vecinos.
- **Naive Bayes** (Lección 5.6): supone características independientes.
- **Random Forest** (Lección 6.2): un ensamble de árboles.

Primero, cada uno por separado, con validación cruzada de 5 pliegues.

In [ ]:
X, y = make_classification(n_samples=1_500, n_features=8, n_informative=5, n_redundant=1,
                           flip_y=0.05, class_sep=0.8, random_state=11)

def miembros():
    return [
        ("logistica", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1_000))),
        ("knn",       make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))),
        ("bayes",     GaussianNB()),
        ("bosque",    RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)),
    ]

resultados = {nombre: cross_val_score(m, X, y, cv=5).mean() for nombre, m in miembros()}
pd.Series(resultados).round(3)

### 🔮 Ejercicio 2 — ¿Duro o suave?
Mira la tabla: hay dos miembros fuertes y dos bastante más débiles. Antes de ejecutar:
1. ¿El **voto duro** de los cuatro superará al mejor miembro?
2. ¿El **voto suave** sacará más o menos que el duro?

✍️ *Mis predicciones:*

In [ ]:
for tipo in ["hard", "soft"]:
    comite = VotingClassifier(miembros(), voting=tipo)
    resultados[f"voto {tipo}"] = cross_val_score(comite, X, y, cv=5).mean()
    print(f"Voto {tipo:<4}: {resultados[f'voto {tipo}']:.3f}")
print(f"Mejor miembro: {max(resultados[n] for n, _ in miembros()):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **No.** El voto duro saca ≈**0.783**, muy por debajo del bosque (≈0.885) y del KNN (≈0.879). Con dos miembros débiles (≈0.63 y ≈0.66) que muchas veces se equivocan **juntos**, a menudo le ganan la votación a los dos buenos (2 contra 2 se desempata a favor de una clase fija). Es lo mismo que viste en la práctica integradora de la Lección 6.1.
2. El voto suave sube mucho, a ≈**0.864**: los modelos buenos suelen estar **más seguros** cuando aciertan, y su seguridad pesa en el promedio. Aun así, **no** supera al bosque solo.
</details>

### ✏️ Ejercicio 3 — Dar más voz a los mejores (`weights`)
Con `weights` puedes darle más peso a algunos miembros. El orden es el de la lista: `[logistica, knn, bayes, bosque]`. Prueba distintas combinaciones y busca la mejor.

In [ ]:
PESOS = [1, 2, 1, 2]       # ✏️ prueba [1, 1, 1, 1], [0, 1, 0, 1], [1, 3, 1, 3]...

comite = VotingClassifier(miembros(), voting="soft", weights=PESOS)
print(f"Voto suave con pesos {PESOS}: {cross_val_score(comite, X, y, cv=5).mean():.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con `[1, 2, 1, 2]` el voto suave sube a ≈**0.879**, al nivel del KNN pero todavía por debajo del bosque. Si sigues dándole más peso a los fuertes, mejora: `[1, 3, 1, 3]` da ≈0.884 y `[0, 1, 0, 1]` (que deja fuera a los débiles) ≈**0.889**, apenas por encima del bosque solo (≈0.885).

Pero fíjate en lo que acabas de hacer: **ajustar pesos a mano** probando combinaciones. ¿No podría un modelo aprender esos pesos solo, y mejor? Esa es justamente la idea del **stacking**.
</details>

### 🔮 Ejercicio 4 — ¿Sirve agregar otro bosque?
Agregamos al voto suave un **segundo** Random Forest, idéntico al primero salvo por la semilla. ¿Cuánto crees que mejora el comité? Para entenderlo, la celda también mide **en qué porcentaje de los ejemplos coinciden** las predicciones de cada par de miembros.

✍️ *Mi predicción:*

In [ ]:
con_gemelo = miembros() + [("bosque_2", RandomForestClassifier(n_estimators=200, random_state=1, n_jobs=-1))]
pred = pd.DataFrame({n: cross_val_predict(m, X, y, cv=5) for n, m in con_gemelo})
print("Coincidencia entre las predicciones de cada par de modelos:")
print(pd.DataFrame({a: {b: np.mean(pred[a] == pred[b]) for b in pred} for a in pred}).round(2))
print(f"\nVoto suave con el gemelo: {cross_val_score(VotingClassifier(con_gemelo, voting='soft'), X, y, cv=5).mean():.3f}"
      f"  (sin él: {resultados['voto soft']:.3f})")

<details><summary><b>👀 Ver respuesta</b></summary>

Las predicciones de los dos bosques coinciden en el **98 %** de los ejemplos: el gemelo casi no aporta opiniones nuevas. Aun así, el voto suave sube un poco (≈0.864 → ≈0.873), pero no por la diversidad sino porque ahora el "equipo bosque" tiene **dos votos**: es casi lo mismo que darle peso 2 al bosque. Fíjate también en el resto de la tabla: KNN y bosque coinciden en el 92 %, y logística y Bayes en el 85 %; en cambio, los fuertes y los débiles solo coinciden en ≈67–72 %.
</details>

---
## 3. 📖 Stacking: el jefe de cocina

En **stacking** (*apilamiento*, Wolpert 1992), los modelos se organizan en dos niveles:

```
Nivel 0 (miembros):    logística   KNN   Bayes   bosque
                           │         │      │       │
                     prob=0.81   0.92   0.30    0.88      ← sus predicciones son las NUEVAS características
                           └─────────┴──────┴───────┘
Nivel 1 (meta-modelo):          regresión logística        ← aprende cuánto confiar en cada uno
                                       │
                                 predicción final
```

El meta-modelo es un modelo normal y corriente, pero sus "características" son **las predicciones de los miembros**. Si aprende que el bosque y el KNN casi siempre aciertan, les dará mucho peso; si aprende que Bayes suele equivocarse, le dará poco o incluso le restará.

### ✋ Ejercicio 5 — Piensa como el jefe
El meta-modelo (una regresión logística) aprendió estos pesos para las probabilidades de cada miembro:

| Miembro | Peso |
|---|---|
| logística | +0.5 |
| KNN | +3.0 |
| Bayes | −0.3 |
| bosque | +3.5 |

1. ¿En quién confía más? ¿Y menos?
2. ¿Qué podría significar un peso **negativo**?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Confía sobre todo en el **bosque** y el **KNN**; la logística aporta poco y Bayes, nada o menos que nada.
2. Que, **una vez que ya sabe lo que dicen los demás**, cuando Bayes está más seguro de la clase 1 conviene **bajar un poco** la probabilidad final. A veces significa que el miembro se equivoca de forma sistemática en los casos donde los demás dudan; a veces es solo ruido. Por eso los pesos de un meta-modelo se leen con cuidado: no son una "calificación" de cada miembro por separado.
</details>

---
## 4. 📖 Qué pasa si… el jefe aprende con trampa

Aquí está **la** trampa del stacking. Para entrenar al meta-modelo necesitamos predicciones de los miembros sobre los datos de entrenamiento. Si usamos las predicciones de miembros que **ya vieron esos mismos datos**, los modelos que memorizan parecerán perfectos y el jefe les creerá ciegamente.

Lo vamos a comprobar con un miembro "tramposo": un **árbol sin límites** (Lección 5.4), que en entrenamiento acierta el 100 % (memoriza) y con datos nuevos es bastante peor que un KNN.

Comparamos dos formas de armar los datos para el meta-modelo:
- **Con trampa:** cada miembro se entrena con todo el entrenamiento y predice **sobre esos mismos datos**.
- **Honesta (fuera de pliegue):** con `cross_val_predict`, cada ejemplo recibe la predicción de un miembro que **no lo vio** (Lección 4.6).

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
nivel0 = {
    "logistica": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1_000)),
    "knn":       make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "arbol":     DecisionTreeClassifier(random_state=0),       # el "tramposo": memoriza
}

def probas(modelos, X_ajuste, y_ajuste, X_pred):
    return np.column_stack([m.fit(X_ajuste, y_ajuste).predict_proba(X_pred)[:, 1] for m in modelos.values()])

meta_X_trampa = probas(nivel0, X_tr, y_tr, X_tr)                       # predicen lo que ya vieron
meta_X_honesta = np.column_stack([cross_val_predict(m, X_tr, y_tr, cv=5, method="predict_proba")[:, 1]
                                  for m in nivel0.values()])          # fuera de pliegue
meta_X_prueba = probas(nivel0, X_tr, y_tr, X_te)                       # para evaluar, siempre igual

### 🔮 Ejercicio 6 — ¿A quién le creerá el jefe tramposo?
Entrenaremos el meta-modelo (una regresión logística) con cada versión y miraremos sus **pesos** y su exactitud en **prueba**.
1. En la versión con trampa, ¿qué miembro recibirá casi todo el peso?
2. ¿Qué versión tendrá mejor exactitud en prueba?

✍️ *Mis predicciones:*

In [ ]:
print("Exactitud de cada miembro en prueba: " +
      ", ".join(f"{n}={np.mean((meta_X_prueba[:, i] >= 0.5) == y_te):.3f}" for i, n in enumerate(nivel0)))
for nombre, meta_X in [("Con trampa", meta_X_trampa), ("Honesta   ", meta_X_honesta)]:
    jefe = LogisticRegression().fit(meta_X, y_tr)
    pesos = {n: round(float(c), 2) for n, c in zip(nivel0, jefe.coef_[0])}
    print(f"{nombre} → pesos del jefe: {pesos} · exactitud del stacking en prueba: {jefe.score(meta_X_prueba, y_te):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con trampa, el jefe le da casi todo el peso al **árbol** (≈7.4, contra ≈2.4 del KNN): en los datos que ya vio, el árbol acertaba el 100 %, ¡parecía infalible!
2. La versión **honesta** gana claramente: ≈**0.867** en prueba, contra ≈0.793 de la versión con trampa, que es exactamente la exactitud del árbol solo (el jefe prácticamente **es** el árbol). La honesta descubre que, fuera de pliegue, el mejor es el KNN (≈0.858 en prueba) y le da el mayor peso (≈6.4), e incluso consigue superarlo un poco.

Por eso `StackingClassifier` de scikit-learn usa **siempre** predicciones fuera de pliegue para entrenar al meta-modelo (parámetro `cv`, por defecto 5 pliegues). Si algún día haces stacking "a mano", esta es la regla de oro.
</details>

### 📖 Stacking con scikit-learn
`StackingClassifier` hace todo lo anterior por ti:
1. Calcula predicciones **fuera de pliegue** de cada miembro (con `cv`).
2. Entrena el meta-modelo (`final_estimator`) con ellas.
3. Reentrena cada miembro con **todo** el entrenamiento para usarlo al predecir.

### 🔮 Ejercicio 7 — ¿Le gana el jefe al mejor miembro?
Volvemos a los cuatro miembros de la Sección 2. ¿Superará el stacking al mejor miembro y al voto suave?

✍️ *Mi apuesta:*

In [ ]:
apilado = StackingClassifier(miembros(), final_estimator=LogisticRegression(), cv=5)
resultados["stacking"] = cross_val_score(apilado, X, y, cv=5).mean()
print(pd.Series(resultados).sort_values(ascending=False).round(3))

apilado.fit(X, y)
print("\nPesos aprendidos por el jefe:", {n: round(float(c), 2) for (n, _), c in zip(miembros(), apilado.final_estimator_.coef_[0])})

<details><summary><b>👀 Ver respuesta</b></summary>

**Sí:** el stacking queda primero con ≈**0.894**, por encima del bosque (≈0.885), del KNN (≈0.879) y del voto suave (≈0.864). La ganancia sobre el mejor miembro es modesta (≈1 punto), pero esta vez el comité **no** fue arrastrado por los débiles.

Mira los pesos del jefe: confía en el **bosque** (≈4.7) y el **KNN** (≈4.1), y a la logística y a Bayes les da pesos **negativos** (≈−1.0 y ≈−0.7). Aprendió solo lo que en el Ejercicio 3 tuvimos que buscar a mano.
</details>

---
## 5. 📖 Hiperparámetros: las perillas del comité

| Hiperparámetro | Dónde | Qué controla | Recomendación |
|---|---|---|---|
| `voting` | Voting | `"hard"` (mayoría de etiquetas) o `"soft"` (promedio de probabilidades) | `"soft"` si todos los miembros dan probabilidades razonables |
| `weights` | Voting | Peso de cada miembro en el voto | Iguales para empezar; ajustarlos a mano es un stacking "pobre" |
| `estimators` | Ambos | Quiénes son los miembros | Pocos (3–6), buenos y **distintos** entre sí |
| `final_estimator` | Stacking | El meta-modelo | Uno **simple**: `LogisticRegression` / `RidgeCV`. Uno complejo sobreajusta |
| `cv` | Stacking | Pliegues para las predicciones fuera de pliegue | 5 (más pliegues = más lento, algo más estable) |
| `stack_method` | Stacking | Qué pasa cada miembro al jefe: `predict_proba`, `decision_function` o `predict` | `"auto"` (usa probabilidades si las hay) |
| `passthrough` | Stacking | Si el jefe ve también las **características originales** | `False`; `True` puede ayudar con un jefe tipo árbol, pero sobreajusta más fácil |

### 🧠 Regla mnemotécnica
> **Miembros distintos, jefe sencillo, y el jefe nunca mira datos que los miembros ya vieron.**

### ✏️ Ejercicio 8 — Un jefe más listo, ¿mejor?
Cambia el meta-modelo por un **Random Forest** y prueba `passthrough=True`. ¿Mejora?

In [ ]:
JEFE = LogisticRegression()                          # ✏️ prueba RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=0)
PASSTHROUGH = False                                  # ✏️ prueba True

apilado = StackingClassifier(miembros(), final_estimator=JEFE, passthrough=PASSTHROUGH, cv=5)
print(f"Stacking (jefe={type(JEFE).__name__}, passthrough={PASSTHROUGH}): {cross_val_score(apilado, X, y, cv=5).mean():.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

En nuestras pruebas, todo queda dentro de **±0.003**: jefe logístico ≈0.894; jefe Random Forest ≈0.891; logístico con `passthrough=True` ≈0.893; Random Forest con `passthrough=True` ≈0.895. Ninguna variante mejora de forma clara, y las complejas tardan más. Un jefe sencillo es suficiente casi siempre: su trabajo es **combinar**, no volver a aprender el problema desde cero.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| El comité rinde **peor** que su mejor miembro | Miembros débiles con el mismo voto que los fuertes | Voto suave, pesos, quitar al débil o stacking |
| Stacking "perfecto" en entrenamiento, malo en prueba | Meta-modelo entrenado con predicciones de miembros que ya vieron esos datos (**fuga**) | Predicciones fuera de pliegue (`StackingClassifier` lo hace) |
| Agregar miembros no mejora nada | Miembros muy parecidos entre sí (p. ej. tres bosques) | Familias distintas; mira la tabla de coincidencias |
| Voto suave raro: un miembro domina | Probabilidades mal calibradas (algunos modelos dan 0.99 con facilidad) | Voto duro, pesos, o calibrar (`CalibratedClassifierCV`) |
| Mejora de 0.2 % con 6 modelos y 10 veces más tiempo | El costo supera el beneficio | Quedarse con el mejor modelo solo; mantenerlo es más barato |
| Con datos pequeños, el stacking varía mucho | El meta-modelo tiene pocos ejemplos para aprender | Jefe muy simple (logística/Ridge con regularización), o voto simple |

> 💡 **En competencias** (Kaggle), los stacking de muchos niveles ganan por milésimas. **En una empresa**, cada modelo extra es código que mantener, monitorear y explicar (Lección 15.1). Pregunta siempre: *¿vale la pena esta mejora?*

---
## 7. 📋 Ficha del modelo

| | Voting | Stacking |
|---|---|---|
| **Tipo** | Ensamble de modelos distintos (clasificación y regresión) | Ensamble de dos niveles (clasificación y regresión) |
| **Intuición** | Jueces que votan o promedian sus notas | Un jefe que aprendió a quién creerle |
| **Cómo aprende** | Entrena cada miembro; combina con mayoría o promedio (ponderado) | Entrena miembros; el meta-modelo aprende de sus predicciones **fuera de pliegue** |
| **Hiperparámetros clave** | `voting`, `weights` | `final_estimator`, `cv`, `passthrough` |
| **✅ Fortalezas** | Simple, rápido, difícil de arruinar si los miembros son parecidos en calidad | Aprende la mejor combinación; suele igualar o superar al mejor miembro |
| **❌ Debilidades** | Los miembros malos arrastran al comité | Más lento (entrena cada miembro `cv`+1 veces), más difícil de explicar, riesgo de fuga si se hace a mano |
| **Úsalo cuando…** | Tienes varios modelos buenos y distintos, de calidad parecida | Quieres exprimir los últimos puntos y tienes datos suficientes |
| **Evítalo cuando…** | Un miembro es claramente mejor que todos | La mejora no compensa la complejidad, o hay muy pocos datos |

## 8. 🧠 Resumen en 5 puntos
1. **Voting** combina modelos distintos por mayoría (**duro**) o promediando probabilidades (**suave**, suele ser mejor).
2. Miembros débiles con el mismo voto **arrastran** al comité: la diversidad ayuda, pero la calidad también cuenta.
3. **Stacking** usa un **meta-modelo** que aprende cuánto confiar en cada miembro.
4. El meta-modelo debe aprender con predicciones **fuera de pliegue**; si no, confiará en los miembros que memorizan (**fuga**).
5. Jefe **sencillo**, miembros **distintos**, y pregúntate siempre si la mejora justifica la complejidad.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈35 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** Tres modelos dan probabilidades 0.6, 0.6 y 0.1 de clase 1. ¿Qué deciden el voto duro y el voto suave?
- a) Duro: 1 · Suave: 1 · b) Duro: 1 · Suave: 0 · c) Duro: 0 · Suave: 1 · d) Duro: 0 · Suave: 0

**2.** ¿Qué son las "características" del meta-modelo en stacking?
- a) Las columnas originales · b) Las predicciones de los modelos del nivel 0 · c) Los hiperparámetros de los miembros

**3.** ¿Por qué el meta-modelo se entrena con predicciones fuera de pliegue?
- a) Para que sea más rápido · b) Para que no sobrevalore a los miembros que memorizan los datos de entrenamiento · c) Porque scikit-learn no permite otra cosa

**4.** ¿Qué meta-modelo es la **mejor primera opción**?
- a) Una red neuronal grande · b) Una regresión logística (o Ridge en regresión) · c) Otro stacking

**5.** Tu comité de 3 bosques con semillas distintas no mejora sobre un solo bosque. ¿Por qué?
- a) Porque tiene número impar · b) Porque los miembros se equivocan casi en los mismos ejemplos · c) Porque los bosques no dan probabilidades

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Duro: dos de tres dicen 1 → **1**. Suave: (0.6 + 0.6 + 0.1) / 3 ≈ 0.43 → **0**. El tercer modelo está muy seguro de que es 0, y los otros dos apenas dudan hacia 1.
2. **b)**
3. **b)** Es la trampa de la Sección 4.
4. **b)** Un jefe sencillo combina; uno complejo sobreajusta con facilidad.
5. **b)** Poca diversidad: es el gemelo del Ejercicio 4 (y los "clones" de la Lección 6.1).
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Si un Random Forest es, en el fondo, un "voto de árboles", ¿en qué se diferencia de un `VotingClassifier` con tres modelos distintos? ¿De dónde sale la diversidad en cada caso?

**B. Caso de análisis 💼** Un equipo presenta un stacking de 7 modelos para aprobar préstamos que mejora el AUC de 0.861 (mejor modelo solo) a 0.866. Entrenarlo tarda 4 horas en vez de 20 minutos, y el regulador pide explicar las decisiones (como en la Lección 5.4). ¿Qué le preguntarías al equipo antes de aprobarlo? ¿Qué recomendarías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** En el bosque todos los miembros son **del mismo tipo** (árboles) y la diversidad sale del **azar** (muestras bootstrap y columnas al azar, Lección 6.2). En el `VotingClassifier`, la diversidad sale de que los miembros **piensan distinto** (rectas, vecinos, probabilidades, preguntas). El bosque usa cientos de miembros baratos; el voting, pocos miembros muy distintos.

**B.** Preguntas útiles:
- ¿La mejora de 0.005 es **real** o cabe en la variación entre pliegues? (¿Cuál es la desviación estándar de la validación cruzada?)
- ¿Cuánto dinero significa en el negocio (Lección 4.8)? ¿Cuántos malos créditos menos por cada 1,000?
- ¿Cómo se explicará una decisión al cliente y al regulador con 7 modelos apilados?
- ¿Quién mantiene y monitorea los 7 modelos?

Recomendación probable: quedarse con el **mejor modelo solo** salvo que la mejora sea claramente significativa y valga dinero; la complejidad tiene costos reales de mantenimiento y explicabilidad.
</details>

### 9.3 🧩 Práctica integradora: comité para predecir la diabetes

Ahora en **regresión**. Los datos `diabetes` de scikit-learn tienen 442 pacientes y 10 medidas (edad, índice de masa corporal, presión…). El objetivo es un número: qué tanto **avanzó la enfermedad** un año después.

Formarás un comité de cuatro modelos y verás si el voto (promedio) y el stacking ayudan… con **pocos** datos.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Los miembros por separado.** Medimos con R² (Lección 4.1) en validación cruzada de 5 pliegues.

In [ ]:
X_d, y_d = load_diabetes(return_X_y=True)

def miembros_reg():
    return [
        ("ridge",   RidgeCV(alphas=np.logspace(-3, 3, 20))),
        ("knn",     make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=20))),
        ("bosque",  RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1)),
        ("boosting", HistGradientBoostingRegressor(learning_rate=0.05, max_iter=200, max_leaf_nodes=7, random_state=0)),
    ]

r2 = {n: cross_val_score(m, X_d, y_d, cv=5, scoring="r2").mean() for n, m in miembros_reg()}
pd.Series(r2).round(3)

✋ ¿Qué modelo gana? ¿Te sorprende, después de todo el módulo?

<details><summary><b>👀 Ver respuesta</b></summary>

Gana **Ridge** (una regresión lineal regularizada, Lección 7.1) con R² ≈**0.478**, por delante de KNN (≈0.448), el bosque (≈0.443) y el boosting (≈0.403). ¡Los ensambles de árboles pierden! Con solo 442 pacientes y una relación casi **lineal** entre las medidas y el avance de la enfermedad, un modelo simple es difícil de superar. Volverás a esta idea en la Lección 6.6.
</details>

**Paso 2 🔮 — El promedio.** `VotingRegressor` simplemente **promedia** las predicciones de los cuatro. ¿Superará a Ridge?

✍️ *Mi apuesta:*

In [ ]:
r2["promedio"] = cross_val_score(VotingRegressor(miembros_reg()), X_d, y_d, cv=5, scoring="r2").mean()
print(f"Promedio de los 4: R² = {r2['promedio']:.3f}  ·  mejor miembro: {max(r2[n] for n, _ in miembros_reg()):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Casi lo iguala, pero no lo supera: ≈**0.476** contra ≈0.478. El promedio suaviza los errores de los cuatro, pero tres de ellos son peores que Ridge y lo "diluyen".
</details>

**Paso 3 ▶️ — El jefe.** Ahora un `StackingRegressor` con un `RidgeCV` como meta-modelo.

In [ ]:
apilado_d = StackingRegressor(miembros_reg(), final_estimator=RidgeCV(alphas=np.logspace(-3, 3, 20)), cv=5)
puntajes = cross_val_score(apilado_d, X_d, y_d, cv=5, scoring="r2")
r2["stacking"] = puntajes.mean()
print(f"Stacking: R² = {puntajes.mean():.3f} (± {puntajes.std():.3f} entre pliegues)")
print(pd.Series(r2).sort_values(ascending=False).round(3))
apilado_d.fit(X_d, y_d)
print("Pesos del jefe:", {n: round(float(c), 2) for (n, _), c in zip(miembros_reg(), apilado_d.final_estimator_.coef_)})

<details><summary><b>👀 Ver respuesta</b></summary>

El stacking empata con Ridge: R² ≈**0.478**. Mira los pesos del jefe: reparte la confianza entre Ridge (≈0.53) y KNN (≈0.50), da un poco al bosque (≈0.25) y un peso negativo al boosting (≈−0.13). Combinar no aportó nada nuevo: el resultado es el mismo que Ridge solo… con cuatro modelos y varias veces más entrenamiento.
</details>

**Paso 4 ✋ — ¿Es real la diferencia?** Mira el "±" del stacking: es cuánto varía el R² de un pliegue a otro. Compáralo con la diferencia entre el stacking y Ridge. ¿Puedes afirmar que uno es mejor que el otro?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

No. El R² del stacking varía **±0.06** de un pliegue a otro, y la diferencia con Ridge es menor que una milésima. Es un **empate técnico**. Cuando dos opciones empatan, elige la más **simple** (la navaja de Occam).
</details>

**Paso 5 ✋ — Recomendación.** Si fueras el analista del hospital, ¿qué modelo pondrías en uso? Escríbelo en dos frases.

<details><summary><b>👀 Ver respuesta</b></summary>

Una respuesta razonable: *"Recomiendo la regresión Ridge: rinde prácticamente igual que cualquier comité, es instantánea de entrenar y cada coeficiente se puede explicar a un médico (p. ej. cuánto suma cada punto de índice de masa corporal). Con solo 442 pacientes, las pequeñas ventajas de los ensambles no son confiables; los reconsideraría si tuviéramos muchos más datos."*

Moraleja del módulo hasta aquí: los ensambles suelen ganar, **pero no siempre**, y medir con honestidad es lo que te permite saber cuándo.
</details>

---
## 📚 Para profundizar
- 📄 **Artículo original:** David Wolpert (1992), *Stacked Generalization*.
- 📘 **Documentación:** scikit-learn, guía de usuario, secciones *Voting Classifier* y *Stacked generalization*.
- 📺 **Video:** "Kaggle Grandmasters explain stacking and blending" y similares: cómo se usan en competencias (y por qué rara vez en producción).

---
⬅️ **Anterior:** [6.4 XGBoost, LightGBM y CatBoost: qué cambia entre ellos](6.4-xgboost-lightgbm-y-catboost.ipynb) · ➡️ **Siguiente:** [6.6 Por qué los ensambles de árboles dominan los datos tabulares](6.6-por-que-los-arboles-dominan-lo-tabular.ipynb)